In [1]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import Point, Polygon, LineString, box
from shapely.ops import unary_union, nearest_points
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set up paths
WATER_MASK_DIR = 'dataset/baringo/processed/sar_water_mask'
OUTPUT_DIR = 'dataset/baringo/processed/expansion_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)

# Display settings
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print("✅ Environment setup complete")
print(f"📁 Water masks directory: {WATER_MASK_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

✅ Environment setup complete
📁 Water masks directory: dataset/baringo/processed/sar_water_mask
📁 Output directory: dataset/baringo/processed/expansion_analysis


## Load and Catalogue Water Masks

We scan the directory for all available water masks, extract their dates from filenames,
and organise them chronologically. Each mask is a binary GeoTIFF where:
- **1** = Water
- **0** = Non-water

In [ ]:
def catalogue_water_masks(mask_dir):
    """
    Scan directory for water mask GeoTIFFs and extract metadata.
    
    Expected filename format: {image_id}.tif
    where image_id contains date information (e.g., S1A_IW_GRDH_20200115_...)
    
    Returns DataFrame with columns: filepath, image_id, date, year
    """
    tif_files = glob.glob(os.path.join(mask_dir, '*.tif'))
    
    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {mask_dir}")
    
    records = []
    for filepath in tif_files:
        image_id = os.path.splitext(os.path.basename(filepath))[0]
        
        # Extract date from Sentinel-1 naming convention
        # Format: S1A_IW_GRDH_1SDV_20200115T030530_...
        try:
            parts = image_id.split('_')
            date_str = None
            for part in parts:
                if len(part) == 8 and part.isdigit():
                    date_str = part
                    break
                elif 'T' in part and part[:8].isdigit():
                    date_str = part[:8]
                    break
            
            if date_str:
                date = pd.to_datetime(date_str, format='%Y%m%d')
            else:
                print(f"⚠️  Could not extract date from: {image_id}")
                continue
        except Exception as e:
            print(f"⚠️  Error parsing date from {image_id}: {e}")
            continue
        
        records.append({
            'filepath': filepath,
            'image_id': image_id,
            'date': date,
            'year': date.year
        })
    
    df = pd.DataFrame(records)
    df = df.sort_values('date').reset_index(drop=True)
    
    return df

# Load catalogue
mask_catalogue = catalogue_water_masks(WATER_MASK_DIR)

print(f"✅ Found {len(mask_catalogue)} water masks")
print(f"📅 Date range: {mask_catalogue['date'].min()} to {mask_catalogue['date'].max()}")
print("\n📋 First 5 entries:")
display(mask_catalogue.head())

print("\n📋 Yearly distribution:")
display(mask_catalogue['year'].value_counts().sort_index())